In [1]:
import rebound
import sys
sys.path.insert(0, '../src')

import sbdynt as sbd

# Default Proper Element Runs for TNOs and Asteroids

Default properties for computing proper elements for TNOs is contained within the same ``run_tno`` function used to initialize and run the machine learning algorithm for TNO resonance occupation.
A similar function, ``run_asteroid`` can be used to similarly run and compute proper elements for asteroids. We demonstrate these functions below for TNO (15760) Albion and the dwarf planet and asteroid (1) Ceres. 

To compute proper elements as well, from the ``run_tno`` function, user should set the boolean variable ``run_proper = True``, which will compute the proper elements and save the results to the ``tno_result`` output. We reccomend also setting ``run_ML=True`` to save time in the event the TNO is a scattering object (if the most common classification is scattering, then by default, the 150 Myr proper element integration is not run)

The bulk of the time required by these functions is actually spent integrating the simulation. The actual computation of the proper elements is comparatively fast. Proper elements can be re-calculated from an existing simulation archive file using the ``analyze_tno_run`` or ``analyze_ast_run`` functions.

In this TNO example, we will still run the machine learning algorithm by including ``run_ML = True``, as this produces a more complex Simulation Archive with varying time resolution. This demonstrates how these complex archives are automatically handled while computing the proper elements from the simulation. We also use ``logfile='screen'``, which will print timestamps and intermediate results to demonstrate and benchmark how long the default runs may take to fully integrate. (Setting ``logfile=<other-string>`` will most of information to a file with that name, minus the printed results, which can be accessed from the returned ``tno_result`` and ``ast_result`` functions.)

See Spencer et al. (2026) for details on how proper elements are calculated

## TNO example -- 150 Myr proper element integration

**NOTE OF CAUTION:** we know we have chosen a classical belt TNO as an example, so we are setting ``integrator='whfast'`` to speed up the integration so this example only takes ~5 minutes to fully execute. **The safest thing to do is to not pass ``integrator`` to ``run_tno``** as that will result in the use of the mercurius hybrid integrator, which can correctly handle a TNO's close encounters with Neptune. If ``run_ML=True`` is set, that short integration will be done with mercurius regardless of the value of ``integrator``, so you will have some warning if the object is actively scattering. But currently stable TNOs can scatter in the future, so use ``integrator='whfast'`` with caution!

The function ``run_tno`` returns a flag (0 if it failed, 1 if it succeeded), the ``tno_result`` class, and the rebound simulation instance ``sim``, which is a snapshot at +75 Myr (the 150 Myr integration is split half backwards and half forwards by default)


In [2]:
flag, tno_result, sim = sbd.run_tno(des='Albion',clones=None,datadir='outputs-from-example-notebooks',
                         archivefile=None,logfile='screen',deletefile=True, 
                         run_ML=True, run_proper=True,integrator='whfast')

Initializing a TNO simulation instance by querying JPL
Clones were not specified, so the default behavior is to return
a best-fit and 3-sigma minimum and maximum semimajor axis clones

SBDB query results saved to outputs-from-example-notebooks/Albion-Mar-18-2026.pkl

simulation epoch: 2456220.5

Rebound simulation initial conditions saved to outputs-from-example-notebooks/Albion-ic.bin

Running TNO ML

Running the 0.5 Myr short classification integration

Running Albion from 0.0 to 500000.0 years 
using mercurius outputting every 50.0 years 
to simulation archivefile outputs-from-example-notebooks/Albion-simarchive.bin
starting at 2026-03-18 10:23:37.764025

finishing at 2026-03-18 10:25:32.560305

Running the classification integration to 10 Myr

Running Albion from 500000.0 to 10000000.0 years 
using mercurius outputting every 1000.0 years 
to simulation archivefile outputs-from-example-notebooks/Albion-simarchive.bin
starting at 2026-03-18 10:25:42.460231

finishing at 2026-03-18 10

In [3]:
print('Albion: ',tno_result.proper_elements.proper_elements)


Albion:  {'a': [43.92886911353194, 43.920765690582186, 43.937687969492295], 'e': [0.07043940738015125, 0.070316041695326, 0.07071086621313116], 'sinI': [0.044411376207242136, 0.04441714745523179, 0.04457266129877757], 'g(rev/yr)': [3.2033027047535085e-07, 3.2065473831380187e-07, 3.204567440349021e-07], 's(rev/yr)': [-3.244978470545344e-07, -3.2541725699217433e-07, -3.233652453264442e-07], 'g("/yr)': [0.41514803053605465, 0.41556854085468725, 0.4153119402692331], 's("/yr)': [-0.42054920978267657, -0.4217407650618579, -0.4190813579430717], 'omega': [0.8576752990299588], 'Omega': [-0.7182776634012898]}


**If you already ran an integration and have the archive file saved, you can use ``analyze_tno_run`` to quickly re-calculate the proper elements**

In [4]:
flag, tno_result = sbd.analyze_tno_run(des='Albion',clones=None,datadir='outputs-from-example-notebooks',
                         archivefile=None,logfile='screen', run_ML=False, run_proper=True)


reading in the simulation archive to determine how many clones there are

Loaded integration for Albion from outputs-from-example-notebooks/Albion-simarchive.bin
simulation is at time -75000000.0 years
integrator is whfast

Found Albion and 2 clones in the simulation

Reading TNO integration for Proper Elements and/or stability indicators

Calculating proper elements

Albion  Proper Element Results from a  150  Myr integration with outputs every 5000 years
# 			 SMA(AU) 	 Ecc    	 Inc(deg) 	 g("/yr) 	 s("/yr)
#Osculating Elements: 	 43.9325 	 0.06938 	 2.18552 	 N/A    	 N/A
#Mean Elements: 	 43.92887 	 0.07062 	 2.83127 	 0.41109 	 -0.321
#Proper Elements: 	 43.92887 	 0.07044 	 2.54542  	 0.41515 	 -0.42055
#Proper Errors: 	 5.319e-05 	 1.947e-04 	 1.821e-02 	 4.044e-04 	 5.281e-03

Clone Proper Elements
Clone #0: 	 43.92077 	 0.07032 	 2.54575  	 0.41557 	 -0.42174
Clone #1: 	 43.93769 	 0.07071 	 2.55467  	 0.41531 	 -0.41908
RMS-Clones+Best-fit: 	 0.00691 	 0.00016 	 7e-05 	 0.000

## Asteroid example -- 10 Myr proper element integration

**NOTE OF CAUTION:** we again set ``integrator='whfast'`` to speed up the integration so this example only takes ~5 minutes to fully execute. Use ``integrator='whfast'`` with caution!

Note that here we set ``clones=0`` to run only the best-fit orbit. Setting ``clones=None`` will do the same 3-sigma clones as in the TNO example abobe, and setting it to some value >0 will produce the specified number of clones sampled in a Guassian manner from the object's covariance matrix in JPL's small body database.

The outputs are the same as in the TNO example above

In [5]:
flag, ast_result, sim = sbd.run_ast(des='Ceres', clones=0, datadir='outputs-from-example-notebooks',
                               logfile='screen',deletefile=True,run_stability=False,integrator='whfast')


Initializing an Asteroid simulation instance by querying JPL for designation: Ceres
SBDB query results saved to outputs-from-example-notebooks/Ceres-Mar-18-2026.pkl

simulation epoch: 2458849.5

Rebound simulation initial conditions saved to outputs-from-example-notebooks/Ceres-ic.bin

Running Asteroid integration for the synthetic
proper elements calculation

Running Ceres from 0.0 to 5000000.0 years 
using whfast outputting every 500.0 years 
to simulation archivefile outputs-from-example-notebooks/Ceres-simarchive.bin
starting at 2026-03-18 10:41:47.529943

finishing at 2026-03-18 10:47:48.321591

Loaded integration for Ceres from outputs-from-example-notebooks/Ceres-ic.bin
simulation is at time 0.0 years
integrator is ias15

Found Ceres and 0 clones in the simulation

Running Ceres from 0.0 to -5000000.0 years 
using whfast outputting every 500.0 years 
to simulation archivefile outputs-from-example-notebooks/Ceres-simarchive.bin
starting at 2026-03-18 10:47:48.332590

finishing at

In [6]:
flag, ast_result = sbd.analyze_ast_run(des='Ceres', clones=None, datadir='outputs-from-example-notebooks',
                               logfile='screen',run_stability=False)


Loaded integration for Ceres from outputs-from-example-notebooks/Ceres-simarchive.bin
simulation is at time -5000000.0 years
integrator is whfast

Found Ceres and 0 clones in the simulation

Reading Asteroid integration

calculating asteroid proper elements

Ceres  Proper Element Results from a  10  Myr integration with outputs every 500 years
# 			 SMA(AU) 	 Ecc    	 Inc(deg) 	 g("/yr) 	 s("/yr)
#Osculating Elements: 	 2.74689 	 0.08057 	 10.60557 	 N/A    	 N/A
#Mean Elements: 	 2.76342 	 0.1181 	 9.76633 	 51.06378 	 -56.78587
#Proper Elements: 	 2.76342 	 0.11504 	 9.64218  	 54.28068 	 -59.22758
#Proper Errors: 	 3.693e-06 	 1.238e-04 	 2.687e-04 	 9.887e-02 	 7.258e-02



## Getting more information from ``tno_result`` and ``ast_result``

We can see the results for the best-fit orbit in an easy way by calling the ``print_results`` function within the proper_element class object inside of the result.

In [7]:
tno_result.proper_elements.print_results()
ast_result.proper_elements.print_results()

Albion  Proper Element Results from a  150  Myr integration with outputs every 5000 years
# 			 SMA(AU) 	 Ecc    	 Inc(deg) 	 g("/yr) 	 s("/yr)
#Osculating Elements: 	 43.9325 	 0.06938 	 2.18552 	 N/A    	 N/A
#Mean Elements: 	 43.92887 	 0.07062 	 2.83127 	 0.41109 	 -0.321
#Proper Elements: 	 43.92887 	 0.07044 	 2.54542  	 0.41515 	 -0.42055
#Proper Errors: 	 5.319e-05 	 1.947e-04 	 1.821e-02 	 4.044e-04 	 5.281e-03

Clone Proper Elements
Clone #0: 	 43.92077 	 0.07032 	 2.54575  	 0.41557 	 -0.42174
Clone #1: 	 43.93769 	 0.07071 	 2.55467  	 0.41531 	 -0.41908
RMS-Clones+Best-fit: 	 0.00691 	 0.00016 	 7e-05 	 0.00017 	 0.00109

Ceres  Proper Element Results from a  10  Myr integration with outputs every 500 years
# 			 SMA(AU) 	 Ecc    	 Inc(deg) 	 g("/yr) 	 s("/yr)
#Osculating Elements: 	 2.74689 	 0.08057 	 10.60557 	 N/A    	 N/A
#Mean Elements: 	 2.76342 	 0.1181 	 9.76633 	 51.06378 	 -56.78587
#Proper Elements: 	 2.76342 	 0.11504 	 9.64218  	 54.28068 	 -59.22758
#Proper 

# Proper Element Outputs

The proper element results are saved to a ``proper_element`` class object within the ``tno_result`` object. 
This class object contains a large number of relevant values and indicators related to the orbital evolution of the small body corresponding to the proper motion over time. These include...

```proper_elements```

```mean_elements```

```osculating_elements```

```proper_errors```

```planet_freqs```

```proper_windows```

### Proper Elements

The proper elements themselves are contained in a dictionary with the same name as the ``proper_element`` class. These can be contrasted with the mean elements, (which are simply the mean of the osculating element time array), and the initial osculating elements, (which represent the orbital elements at time ``t=0`` in the simulation).

We note that these mean elements are not be confused with the ``mean`` indicator, which is used to identify objects which experience chaotic motion or long-term periodic evolution such that the synethic proper element cannot be accurately computed for the small-body in the given integration. 
We will discuss the ``mean`` indicator, as well as other useful indicators, in more detail in the ``proper_elements_advanced`` notebook. 

These three outputs contain the semi-major axis, eccentricity, inclination, as well as the proper and mean precession rates, ``g`` and ``s``, in the case of the ``proper_elements`` and ``mean_elements`` variables. The ``osculating_elements`` variable instead reports the initial ``omega`` and ``Omega`` values at the ``t=0`` epoch.

The proper and mean ``g`` and ``s`` frequencies are reported in both revolutions/year (the units of measurement used by the filtering, the direct result from the filter) and arcseconds/year (the typically reported value for the proper precession rates).

In [8]:
print('Albion: ',tno_result.proper_elements.proper_elements)
#print('\nCeres:', ast_result.proper_elements.proper_elements)


Albion:  {'a': [43.92886911353194, 43.920765690582186, 43.937687969492295], 'e': [0.07043940738015125, 0.070316041695326, 0.07071086621313116], 'sinI': [0.044411376207242136, 0.04441714745523179, 0.04457266129877757], 'g(rev/yr)': [3.2033027047535085e-07, 3.2065473831380187e-07, 3.204567440349021e-07], 's(rev/yr)': [-3.244978470545344e-07, -3.2541725699217433e-07, -3.233652453264442e-07], 'g("/yr)': [0.41514803053605465, 0.41556854085468725, 0.4153119402692331], 's("/yr)': [-0.42054920978267657, -0.4217407650618579, -0.4190813579430717], 'omega': [0.8576752990299588], 'Omega': [-0.7182776634012898]}


In [9]:
print('Albion:', tno_result.proper_elements.mean_elements)
print('\nCeres:', ast_result.proper_elements.mean_elements)

Albion: {'a': [43.92886911353193, 43.920765690582186, 43.93768796949228], 'e': [0.07061900302409709, 0.07047485108138321, 0.07088419843387968], 'sinI': [0.04939480095426884, 0.04936373061881539, 0.049510836831941504], 'g(rev/yr)': [3.171966755039007e-07, 3.1679233429152504e-07, 3.177788346585308e-07], 's(rev/yr)': [-2.476842526486315e-07, -2.4851495652447944e-07, -2.4729841841155693e-07], 'g("/yr)': [0.4110868914530553, 0.41056286524181645, 0.41184136971745594], 's("/yr)': [-0.3209987914326264, -0.32207538365572536, -0.32049875026137775]}

Ceres: {'a': [2.76342167496004], 'e': [0.1180950385317162], 'sinI': [0.16963046470501486], 'g(rev/yr)': [3.940106413675179e-05], 's(rev/yr)': [-4.3816255727502476e-05], 'g("/yr)': [51.06377912123031], 's("/yr)': [-56.78586742284321]}


In [10]:
print('Albion:',tno_result.proper_elements.osculating_elements)
print('\nCeres:',ast_result.proper_elements.osculating_elements)

Albion: {'a': [43.93250143260439, 43.92429713205721, 43.94120869057395], 'e': [0.06937743939002076, 0.06924305870788125, 0.06951644409153306], 'I': [0.038144454843896995, 0.0381442505630826, 0.03814524197650658], 'omega': [0.04935981885407337, 0.049020986337060535, 0.050000504016175285], 'Omega': [-0.010239840041737528, -0.010241744565488558, -0.01022987998146208]}

Ceres: {'a': [2.74688581425474], 'e': [0.08056521544696876], 'I': [0.1851020566317099], 'omega': [1.2361490429858932], 'Omega': [1.4003739058029678]}


### Proper Element Uncertainties

The proper element uncertainties are contained in the ``proper_errors`` dictionary.

In [11]:
print('Albion:', tno_result.proper_elements.proper_errors)
print('\nCeres:',ast_result.proper_elements.proper_errors)

Albion: {'RMS_a': [5.319110891477913e-05, 9.450337128231006e-06, 0.0002425346212776684], 'RMS_e': [0.00019465053387827285, 0.00022743002268939754, 0.00023204508479955212], 'RMS_sinI': [0.00031780092944478467, 0.00027731221328436385, 0.0002980527923550516], 'RMS_g(rev/yr)': [3.1205195187524303e-10, 6.251995631518795e-10, 4.3380489700055097e-10], 'RMS_s(rev/yr)': [4.074621313368901e-09, 4.819295828519004e-09, 3.0854631984575203e-09], 'RMS_g("/yr)': [0.00040441932963031495, 0.0008102586338448357, 0.000562211146512714], 'RMS_s("/yr)': [0.005280709222126095, 0.006245807393760629, 0.003998760305200947]}

Ceres: {'RMS_a': [3.6927019545968875e-06], 'RMS_e': [0.00012380153345236652], 'RMS_sinI': [4.689196117264828e-06], 'RMS_g(rev/yr)': [7.628518495769389e-08], 'RMS_s(rev/yr)': [5.600328511406815e-08], 'RMS_g("/yr)': [0.09886559970517127], 'RMS_s("/yr)': [0.07258025750783233]}


### Planetary Frequencies

Users interested in the identified planetary secular frequencies can retrieve these from the ``planets`` dictionary. These are only reported in the units of rev/yr. Users can retrieve the "/yr units by multipying these results by 1296000.

In [12]:
print('Albion:', tno_result.proper_elements.planet_freqs)
print('\nCeres:', ast_result.proper_elements.planet_freqs)

Albion: {'g5': 3.273690577014656e-06, 'g6': 2.1786512954960542e-05, 'g7': 2.3812103203317446e-06, 'g8': 5.199958109458286e-07, 's6': -2.0326125357009747e-05, 's7': -2.3093478234800085e-06, 's8': -5.333297223319403e-07}

Ceres: {'g5': 3.299919347686973e-06, 'g6': 2.1799952684428964e-05, 'g7': 2.3998136076643332e-06, 'g8': 4.999354125342965e-07, 's6': -2.030384012798331e-05, 's7': -2.3000030066524188e-06, 's8': -5.060633131971425e-07, 'g2': 5.5120420968311885e-06, 'g3': 1.3607986408556482e-05, 'g4': 1.311427566334707e-05, 's3': -1.445635370404582e-05, 's4': -1.3697570469382686e-05, 's2': -4.793555853266439e-06}


### Window Proper Elements

Users may also access the proper element computed for each of the windows. This could be useful in cases of objects which experience chaotic or scattering motion, if the users wishes to see the proper element before such events. 

In [13]:
print('Albion:', tno_result.proper_elements.proper_windows)
print('\nCeres:', ast_result.proper_elements.proper_windows)

Albion: {'a_win': array([43.92880024, 43.92883829, 43.92888318, 43.92892818, 43.92893814]), 'e_win': array([0.07021873, 0.07039975, 0.07037384, 0.07015245, 0.07021022]), 'sinI_win': array([0.04408636, 0.04454724, 0.0443236 , 0.04395335, 0.04400714]), 'g_win': array([3.20017712e-07, 3.20021700e-07, 3.20021300e-07, 3.20014285e-07,
       3.20016162e-07]), 's_win': array([-3.20180727e-07, -3.21064128e-07, -3.20732149e-07, -3.20088893e-07,
       -3.20142920e-07])}

Ceres: {'a_win': array([2.76342321, 2.76341905, 2.76341429, 2.7634222 , 2.76341965]), 'e_win': array([0.11519433, 0.11522566, 0.11515439, 0.11502146, 0.11497271]), 'sinI_win': array([0.16749849, 0.16749359, 0.16749162, 0.16748975, 0.16748659]), 'g_win': array([4.18067698e-05, 4.18068312e-05, 4.18069688e-05, 4.18071123e-05,
       4.18071063e-05]), 's_win': array([-4.57760422e-05, -4.57651990e-05, -4.57487075e-05, -4.57419428e-05,
       -4.57409626e-05])}


More advanced outputs are further discussed in the ``proper_elements_advanced.ipynb`` file, which include flags related to chaos, the amplitude of secular terms, as well as some indicators which may be hekpful for identifying secualr resonance occupation. 

In [14]:
reflag, times, sb_elems, planet_elems, clone_elems, small_planets_flag = \
                sbd.read_archive_for_pe(des='Albion',archivefile=tno_result.archivefile,
                                clones=tno_result.clones,logfile=tno_result.logfile, 
                                object_type = tno_result.object_type)
